# Krishok AI ? Rice Disease Classification: MobileNetV3-Large

Google Colab training pipeline for the approved six-class RiceLeafDiseaseBD split. It first rebuilds a clean runtime copy from the Drive dataset using the approved CSV manifests. Preflight verifies manifest SHA-256, exact relative path inventories, per-image MD5, full image decoding, class mapping, split counts, duplicate-group isolation, and exclusion of unresolved cross-class duplicate groups. Any mismatch raises an error and stops execution; no corrupt image is skipped.

Training uses ImageNet-pretrained MobileNetV3-Large, 224?224 inputs, AdamW, a validation-loss scheduler, early stopping, and best/last checkpoints. Training, exports, and the one-time final test evaluation are explicit opt-ins and default off. Test rows never enter training or tuning; the optional final test cell is for frozen reporting only.

In [ ]:
%pip -q install pandas scikit-learn matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, platform, random, shutil, time
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V3_Large_Weights

SEED = 42
IMG_SIZE = 224
MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 8
BATCH_SIZE = 32  # If CUDA runs out of memory, set 16 (or 8) and restart the training cell.
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
START_TRAINING = False
RUN_EXPORTS = False
RUN_FINAL_TEST_EVALUATION = False  # Final frozen report only; never use results for tuning.

DRIVE_ROOT = Path('/content/drive/MyDrive/Krishok_AI')
SOURCE_DATA_ROOT = DRIVE_ROOT / 'data/processed/rice_classification'
MANIFEST_DIR = SOURCE_DATA_ROOT / 'manifests'
RUNTIME_DATA_ROOT = Path('/content/rice_classification_verified')
RUNTIME_STAGE_ROOT = Path('/content/rice_classification_verified.building')
EXPERIMENT_DIR = DRIVE_ROOT / 'AI/experiments/MobileNetV3_Rice'
RUN_DIR = EXPERIMENT_DIR
TRAIN_CSV = MANIFEST_DIR / 'train.csv'
VAL_CSV = MANIFEST_DIR / 'val.csv'
TEST_CSV = MANIFEST_DIR / 'test.csv'
CLASS_MAP_PATH = MANIFEST_DIR / 'class_mapping.json'
CROSS_REVIEW_PATH = MANIFEST_DIR / 'cross_class_review.csv'
EXPECTED_COUNTS = {'train': 6827, 'val': 1463, 'test': 1463}
EXPECTED_CLASS_TO_ID = {
    'Blast': 0, 'Brown spot': 1, 'Healthy': 2,
    'Leaf smut': 3, 'Rice Tungro': 4, 'Sheath blight': 5,
}
EXPECTED_MANIFEST_SHA256 = {
    'train.csv': 'baf1004abaa44f3ddddd7bde3abc5cefc15d06904e4b441eb0dcf87ea41d0b0d',
    'val.csv': '60706b05853801deb524ea078bccc7c62ef4a23029008d7b28a649afed2d0145',
    'test.csv': '91fdd26f22243861fcf1eea53b3312d1e1bcdb5800ffa822f56ab486b3d59d9f',
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before proceeding.'
device = torch.device('cuda:0')
print({'python': platform.python_version(), 'torch': torch.__version__,
       'torchvision': __import__('torchvision').__version__, 'gpu': torch.cuda.get_device_name(0)})

## 1. Strict dataset verification and runtime rebuild

This preflight is intentionally strict and can take time because every approved image is fully decoded and checked against the manifest MD5. It reads `test.csv` only to validate and rebuild the frozen test inventory. Test images are not loaded into a training/validation dataset. A fresh runtime directory is required; if it already exists, the notebook stops rather than deleting or overwriting it.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

class_map = json.loads(CLASS_MAP_PATH.read_text(encoding='utf-8'))
classes = class_map.get('classes', [])
class_to_id = {item['class_label']: int(item['class_id']) for item in classes}
assert class_to_id == EXPECTED_CLASS_TO_ID, f'Approved class mapping mismatch: {class_to_id}'
assert class_map.get('uses_yolo_annotation_class_ids') is False, 'Classification labels must come from the approved folder mapping.'
manifest_paths = {'train': TRAIN_CSV, 'val': VAL_CSV, 'test': TEST_CSV}
for split, path in manifest_paths.items():
    assert path.is_file(), f'Missing approved manifest: {path}'
    actual_sha = sha256_file(path)
    expected_sha = EXPECTED_MANIFEST_SHA256[f'{split}.csv']
    assert actual_sha == expected_sha, f'{split}.csv SHA-256 mismatch: expected {expected_sha}, got {actual_sha}'

frames = {split: pd.read_csv(path, dtype={'image_path':str, 'class_label':str}) for split,path in manifest_paths.items()}
for split, frame in frames.items():
    assert len(frame) == EXPECTED_COUNTS[split], f'{split} count mismatch: expected {EXPECTED_COUNTS[split]}, got {len(frame)}'
    assert frame['image_path'].is_unique, f'Duplicate relative paths in {split}.csv'
    assert set(frame['class_label']) == set(class_to_id), f'Not all six classes present in {split}'
    mapped = frame['class_label'].map(class_to_id).astype(int)
    assert frame['class_id'].astype(int).equals(mapped), f'class_id/class_label mismatch in {split}'
    assert frame['image_path'].str.startswith(split + '/').all(), f'Unexpected split path prefix in {split}.csv'

# No exact duplicate group may cross splits; unresolved cross-class groups must remain excluded.
split_groups = {s:set(f['duplicate_group_id'].astype(str)) for s,f in frames.items()}
for a,b in [('train','val'),('train','test'),('val','test')]:
    assert not (split_groups[a] & split_groups[b]), f'Exact duplicate group crosses {a}/{b}'
review = pd.read_csv(CROSS_REVIEW_PATH)
unresolved = set(review.loc[review.review_status == 'unresolved_excluded', 'group_id'].astype(str))
assert not (unresolved & set.union(*split_groups.values())), 'An unresolved cross-class group appears in an approved split.'

assert not RUNTIME_DATA_ROOT.exists(), (
    f'Runtime destination already exists: {RUNTIME_DATA_ROOT}. Preserve it for diagnosis and choose a new empty runtime path.'
)
assert not RUNTIME_STAGE_ROOT.exists(), (
    f'Previous incomplete staging directory exists: {RUNTIME_STAGE_ROOT}. Preserve and inspect it, then choose a new staging path.'
)
RUNTIME_STAGE_ROOT.mkdir(parents=True, exist_ok=False)
(RUNTIME_STAGE_ROOT / 'manifests').mkdir()
for path in [*manifest_paths.values(), CLASS_MAP_PATH, CROSS_REVIEW_PATH]:
    shutil.copy2(path, RUNTIME_STAGE_ROOT / 'manifests' / path.name)

inventory_report = []
for split, frame in frames.items():
    expected = set(frame['image_path'])
    split_root = SOURCE_DATA_ROOT / split
    actual = {p.relative_to(SOURCE_DATA_ROOT).as_posix() for p in split_root.rglob('*')
              if p.is_file()}
    assert actual == expected, (
        f'{split} source inventory mismatch: missing={sorted(expected-actual)[:10]}, '
        f'extra={sorted(actual-expected)[:10]}'
    )
    for row in frame.itertuples(index=False):
        source = SOURCE_DATA_ROOT / row.image_path
        destination = RUNTIME_STAGE_ROOT / row.image_path
        assert source.suffix.lower() in IMAGE_SUFFIXES, f'Unsupported manifest image format: {source}'
        assert source.is_file(), f'Manifest image is missing: {source}'
        try:
            with Image.open(source) as image:
                image.load()  # Full decode; never silently skip a corrupt file.
                assert image.width > 0 and image.height > 0, f'Invalid dimensions: {source}'
        except Exception as exc:
            raise RuntimeError(f'Image decode failed; aborting without skipping: {source}: {exc}') from exc
        actual_md5 = hashlib.md5(source.read_bytes()).hexdigest()
        assert actual_md5 == row.md5, f'Image MD5 mismatch for {row.image_path}: {actual_md5} != {row.md5}'
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied_md5 = hashlib.md5(destination.read_bytes()).hexdigest()
        assert copied_md5 == row.md5, f'Runtime copy MD5 mismatch for {row.image_path}'
    copied = {p.relative_to(RUNTIME_DATA_ROOT).as_posix() for p in (RUNTIME_STAGE_ROOT / split).rglob('*')
              if p.is_file()}
    assert copied == expected, f'{split} runtime inventory mismatch: missing={sorted(expected-copied)[:10]}, extra={sorted(copied-expected)[:10]}'
    inventory_report.append({'split':split,'manifest_count':len(frame),'source_count':len(actual),'runtime_count':len(copied),'missing':0,'extra':0})

# Copy provenance metadata too; verify the runtime manifest copies still have the pinned hashes.
for split in ('train','val','test'):
    runtime_manifest = RUNTIME_STAGE_ROOT / 'manifests' / f'{split}.csv'
    assert sha256_file(runtime_manifest) == EXPECTED_MANIFEST_SHA256[f'{split}.csv']

RUNTIME_STAGE_ROOT.rename(RUNTIME_DATA_ROOT)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
report_lines = [
    '# Dataset Integrity Verification Report', '',
    '**Verification status:** PASS (runtime-generated by this notebook preflight)', '',
    'All three approved manifests matched their pinned SHA-256 hashes. Source and rebuilt Colab runtime relative image paths matched each manifest exactly. Every image was fully decoded; every source and copied image MD5 matched its manifest entry. No file was silently skipped.', '',
    '| Split | Manifest | Source files | Runtime files | SHA-256 |', '|---|---:|---:|---:|---|'
]
for item in inventory_report:
    split=item['split']; digest=EXPECTED_MANIFEST_SHA256[f'{split}.csv']
    report_lines.append(f"| {split} | {item['manifest_count']} | {item['source_count']} | {item['runtime_count']} | `{digest}` |")
report_lines += ['', 'Class mapping: ' + ', '.join(f'{name}={idx}' for name,idx in class_to_id.items()),
                 '', 'Test files were copied and integrity-checked for a frozen final evaluation, but were not placed in a DataLoader during training or validation.']
(EXPERIMENT_DIR / 'dataset_integrity_verification_report.md').write_text('\n'.join(report_lines)+'\n', encoding='utf-8')

# Training loaders use only verified runtime copies. Test rows remain outside the training code path.
DATA_ROOT = RUNTIME_DATA_ROOT
train_df, val_df = frames['train'], frames['val']
print({'integrity':'PASS','runtime_dataset':str(RUNTIME_DATA_ROOT),'inventory':inventory_report,
       'class_mapping':class_to_id,'unresolved_cross_class_groups_excluded':len(unresolved),
       'manifest_sha256':EXPECTED_MANIFEST_SHA256})

## 2. Train and validation loaders

Training augmentation is random resized crop plus horizontal flip at 224?224, then ImageNet normalization. Validation uses the MobileNetV3-Large ImageNet preset (resize/center-crop, tensor conversion, normalization). The test split is not included in either loader.

In [ ]:
weights = MobileNet_V3_Large_Weights.DEFAULT
eval_transform = weights.transforms()
mean, std = eval_transform.mean, eval_transform.std
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])
print('ImageNet mean:', mean, 'std:', std)
print('Validation preprocessing:', eval_transform)

class ManifestImageDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = DATA_ROOT / row.image_path
        try:
            with Image.open(path) as image:
                image.load()
                tensor = self.transform(image.convert('RGB'))
        except Exception as exc:
            raise RuntimeError(f'Image read failed during training; no sample was skipped: {path}: {exc}') from exc
        return tensor, int(row.class_id)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(ManifestImageDataset(train_df, train_transform), batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker,
    generator=generator, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(ManifestImageDataset(val_df, eval_transform), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0))

## 3. MobileNetV3-Large training (explicit opt-in)

Best checkpoint selection, scheduler decisions, and early stopping use validation loss only. For CUDA out-of-memory, reduce `BATCH_SIZE` to 16 or 8, restart the training cell, and use a new experiment directory if a prior run has started. The test set is never read by this training cell.

In [ ]:
def build_model(pretrained=True):
    model = models.mobilenet_v3_large(weights=weights if pretrained else None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, len(class_to_id))
    return model

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_count, correct = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += float(loss.item()) * labels.size(0)
            total_count += labels.size(0)
            correct += int((logits.argmax(1) == labels).sum().item())
    return total_loss / total_count, correct / total_count

if START_TRAINING:
    if any((RUN_DIR / name).exists() for name in ['best_model.pt','last_model.pt','training_history.csv']):
        raise FileExistsError(f'An experiment already exists; choose a new Drive experiment directory: {RUN_DIR}')
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    model = build_model(pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    history, best_val_loss, best_epoch, stale_epochs = [], float('inf'), 0, 0
    best_path, last_path = RUN_DIR/'best_model.pt', RUN_DIR/'last_model.pt'
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy = run_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)
        learning_rate = optimizer.param_groups[0]['lr']
        history.append({'epoch':epoch,'train_loss':train_loss,'train_accuracy':train_accuracy,
                        'val_loss':val_loss,'val_accuracy':val_accuracy,'learning_rate':learning_rate})
        checkpoint_data = {'model_state_dict':model.state_dict(),'class_to_id':class_to_id,
                           'epoch':epoch,'val_loss':val_loss,'seed':SEED,
                           'train_manifest_sha256':EXPECTED_MANIFEST_SHA256['train.csv'],
                           'val_manifest_sha256':EXPECTED_MANIFEST_SHA256['val.csv']}
        torch.save(checkpoint_data, last_path)
        print(f'Epoch {epoch:02d}/{MAX_EPOCHS}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_accuracy={val_accuracy:.4f} lr={learning_rate:.2e}')
        if val_loss < best_val_loss - 1e-4:
            best_val_loss, best_epoch, stale_epochs = val_loss, epoch, 0
            torch.save(checkpoint_data, best_path)
        else:
            stale_epochs += 1
        if stale_epochs >= EARLY_STOP_PATIENCE:
            print(f'Early stopping after {epoch} epochs; best epoch was {best_epoch}.')
            break
    pd.DataFrame(history).to_csv(RUN_DIR/'training_history.csv', index=False)
    environment = {'model':'torchvision.mobilenet_v3_large','weights':str(weights),'seed':SEED,
        'imgsz':IMG_SIZE,'batch_size':BATCH_SIZE,'epochs_max':MAX_EPOCHS,
        'patience':EARLY_STOP_PATIENCE,'torch':torch.__version__,
        'torchvision':__import__('torchvision').__version__,'cuda':torch.version.cuda,
        'gpu':torch.cuda.get_device_name(0),'best_epoch':best_epoch,'class_to_id':class_to_id,
        'train_manifest_sha256':EXPECTED_MANIFEST_SHA256['train.csv'],
        'val_manifest_sha256':EXPECTED_MANIFEST_SHA256['val.csv'],
        'test_manifest_sha256':EXPECTED_MANIFEST_SHA256['test.csv'],
        'test_used_during_training_or_tuning':False}
    (RUN_DIR/'environment.json').write_text(json.dumps(environment,indent=2),encoding='utf-8')
    print('Best checkpoint:',best_path,'Last checkpoint:',last_path)
else:
    print('Training not started. Set START_TRAINING=True only after reviewing the passing preflight.')

## 4. Validation metrics, confusion matrix, loss curves, model size, and CPU/GPU latency

After training, this section evaluates the best checkpoint on the approved validation split. It writes accuracy, macro precision/recall/F1, per-class metrics, a validation confusion matrix, parameter count, checkpoint size, and separate batch-one CPU and GPU forward-pass latency. Latency uses preprocessed 224?224 validation tensors and excludes image decode/transforms. No test examples are read here.

In [ ]:
import os
if (RUN_DIR/'training_history.csv').is_file() and (RUN_DIR/'best_model.pt').is_file():
    import matplotlib.pyplot as plt
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

    history_df=pd.read_csv(RUN_DIR/'training_history.csv')
    fig,ax=plt.subplots(figsize=(8,5))
    ax.plot(history_df.epoch,history_df.train_loss,label='Train loss')
    ax.plot(history_df.epoch,history_df.val_loss,label='Validation loss')
    ax.set(xlabel='Epoch',ylabel='Cross-entropy loss',title='MobileNetV3-Large training and validation loss')
    ax.grid(True); ax.legend(); fig.tight_layout()
    fig.savefig(RUN_DIR/'training_validation_loss_curves.png',dpi=160); plt.show()

    class_names=[name for name,_ in sorted(class_to_id.items(),key=lambda kv:kv[1])]
    checkpoint=torch.load(RUN_DIR/'best_model.pt',map_location='cpu',weights_only=True)
    assert checkpoint['class_to_id']==class_to_id, 'Checkpoint class mapping differs from the approved mapping.'
    eval_model=build_model(pretrained=False).to(device).eval()
    eval_model.load_state_dict(checkpoint['model_state_dict'])
    val_true=[]; val_pred=[]
    # val_loader is deterministic and contains only approved val.csv rows.
    with torch.inference_mode():
        for images,labels in val_loader:
            logits=eval_model(images.to(device,non_blocking=True))
            val_pred.extend(logits.argmax(1).cpu().tolist())
            val_true.extend(labels.tolist())
    y_true=np.asarray(val_true,dtype=int); y_pred=np.asarray(val_pred,dtype=int)
    assert len(y_true)==len(val_df)==1463, f'Validation evaluation count mismatch: {len(y_true)}'
    per_class=precision_recall_fscore_support(y_true,y_pred,labels=list(range(6)),average=None,zero_division=0)
    macro_p,macro_r,macro_f1,_=precision_recall_fscore_support(y_true,y_pred,labels=list(range(6)),average='macro',zero_division=0)
    report=classification_report(y_true,y_pred,labels=list(range(6)),target_names=class_names,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(RUN_DIR/'validation_classification_report.csv',index=True,index_label='class')
    cm=confusion_matrix(y_true,y_pred,labels=list(range(6)))
    fig,ax=plt.subplots(figsize=(9,7)); im=ax.imshow(cm,cmap='Blues')
    ax.set(xticks=range(6),yticks=range(6),xticklabels=class_names,yticklabels=class_names,
           xlabel='Predicted class',ylabel='True class',title='MobileNetV3-Large validation confusion matrix')
    plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
    for i in range(6):
        for j in range(6): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()*.55 else 'black')
    fig.colorbar(im,ax=ax,label='Images'); fig.tight_layout()
    fig.savefig(RUN_DIR/'validation_confusion_matrix.png',dpi=180); plt.show()

    # Benchmark forward-pass latency on the same preprocessed validation images for CPU and GPU.
    bench_rows=val_df.iloc[:min(50,len(val_df))]
    bench_tensors=[]
    for relpath in bench_rows.image_path:
        with Image.open(DATA_ROOT/relpath) as image:
            image.load(); bench_tensors.append(eval_transform(image.convert('RGB')))
    bench_cpu=torch.stack(bench_tensors)
    def measure_forward_latency(model, input_tensor, target_device, warmup=5, iterations=50):
        model=model.to(target_device).eval()
        inputs=input_tensor.to(target_device)
        def sync():
            if target_device.type=='cuda': torch.cuda.synchronize(target_device)
        with torch.inference_mode():
            for i in range(warmup): _=model(inputs[i % len(inputs)].unsqueeze(0))
        sync(); samples=[]
        with torch.inference_mode():
            for i in range(iterations):
                sample=inputs[i % len(inputs)].unsqueeze(0)
                sync(); start=time.perf_counter(); _=model(sample); sync()
                samples.append((time.perf_counter()-start)*1000)
        return {'device':str(target_device),'batch_size':1,'warmup_iterations':warmup,
                'timed_iterations':iterations,'mean_ms':float(np.mean(samples)),
                'median_ms':float(np.median(samples)),'p95_ms':float(np.percentile(samples,95))}
    cpu_threads=max(1,min(4,os.cpu_count() or 1)); torch.set_num_threads(cpu_threads)
    cpu_model=build_model(pretrained=False)
    cpu_model.load_state_dict(checkpoint['model_state_dict'])
    cpu_latency=measure_forward_latency(cpu_model,bench_cpu,torch.device('cpu'))
    gpu_latency=measure_forward_latency(eval_model,bench_cpu,device)
    size_bytes=(RUN_DIR/'best_model.pt').stat().st_size
    params=sum(param.numel() for param in eval_model.parameters())
    val_summary={'split':'validation','images':len(y_true),'accuracy':float(accuracy_score(y_true,y_pred)),
        'macro_precision':float(macro_p),'macro_recall':float(macro_r),'macro_f1':float(macro_f1),
        'weighted_f1':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(6)),average='weighted',zero_division=0)[2]),
        'class_to_id':class_to_id,'train_manifest_sha256':EXPECTED_MANIFEST_SHA256['train.csv'],
        'val_manifest_sha256':EXPECTED_MANIFEST_SHA256['val.csv'],'test_used':False,
        'parameters':int(params),'best_checkpoint_bytes':size_bytes,
        'best_checkpoint_MiB':size_bytes/(1024**2),'cpu_inference_latency':cpu_latency,
        'gpu_inference_latency':gpu_latency,'latency_note':'Forward-only, batch 1, fixed preprocessed 224x224 RGB inputs; decode and preprocessing excluded.'}
    (RUN_DIR/'validation_metrics.json').write_text(json.dumps(val_summary,indent=2),encoding='utf-8')
    latency_summary={'validation_images_for_benchmark':len(bench_tensors),'input_resolution':[224,224],
                     'cpu':cpu_latency,'gpu':gpu_latency,'parameters':int(params),
                     'best_checkpoint_bytes':size_bytes,'best_checkpoint_MiB':size_bytes/(1024**2),
                     'measurement':'Forward-only batch-one inference; preprocessing excluded.'}
    (RUN_DIR/'inference_latency.json').write_text(json.dumps(latency_summary,indent=2),encoding='utf-8')
    print(json.dumps(val_summary,indent=2))
else:
    print('Validation metrics, curves, and latency reports will be generated after training completes.')

## 5. One-time final frozen test report (explicit opt-in)

Run only after training is complete and the model/checkpoint is frozen. This section evaluates test data once for reporting, writes per-class metrics, confusion matrix, macro F1, weighted F1, and test inference latency, and must not be used to tune or select a model. `RUN_FINAL_TEST_EVALUATION` defaults to `False`.

In [ ]:
if RUN_FINAL_TEST_EVALUATION:
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
    import matplotlib.pyplot as plt
    assert (RUN_DIR/'best_model.pt').is_file() and (RUN_DIR/'environment.json').is_file(), 'Complete training first.'
    env=json.loads((RUN_DIR/'environment.json').read_text(encoding='utf-8'))
    assert env['train_manifest_sha256']==EXPECTED_MANIFEST_SHA256['train.csv']
    assert env['val_manifest_sha256']==EXPECTED_MANIFEST_SHA256['val.csv']
    test_df=frames['test'].reset_index(drop=True)  # used only for final, frozen reporting
    final_model=build_model(pretrained=False).to(device).eval()
    ckpt=torch.load(RUN_DIR/'best_model.pt',map_location=device,weights_only=True)
    assert ckpt['class_to_id']==class_to_id
    final_model.load_state_dict(ckpt['model_state_dict'])
    y_true=test_df.class_id.astype(int).to_numpy(); y_pred=[]; latencies=[]
    class_names=[name for name,_ in sorted(class_to_id.items(),key=lambda kv:kv[1])]
    # One deterministic pass. No gradients, updates, or threshold/hyperparameter selection.
    for relpath in test_df.image_path:
        path=DATA_ROOT/relpath
        torch.cuda.synchronize(); t0=time.perf_counter()
        with Image.open(path) as image:
            image.load(); tensor=eval_transform(image.convert('RGB')).unsqueeze(0).to(device)
        with torch.inference_mode(): logits=final_model(tensor)
        torch.cuda.synchronize(); latencies.append((time.perf_counter()-t0)*1000)
        y_pred.append(int(logits.argmax(1).item()))
    y_pred=np.asarray(y_pred); cm=confusion_matrix(y_true,y_pred,labels=list(range(len(class_names))))
    metrics=classification_report(y_true,y_pred,labels=list(range(len(class_names))),target_names=class_names,
                                  output_dict=True,zero_division=0)
    pd.DataFrame(metrics).T.to_csv(RUN_DIR/'test_classification_report.csv',index=True,index_label='class')
    fig,ax=plt.subplots(figsize=(9,7)); im=ax.imshow(cm,cmap='Blues')
    ax.set(xticks=range(6),yticks=range(6),xticklabels=class_names,yticklabels=class_names,
           xlabel='Predicted class',ylabel='True class',title='MobileNetV3-Large frozen test confusion matrix')
    plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
    for i in range(6):
        for j in range(6): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()*.55 else 'black')
    fig.colorbar(im,ax=ax,label='Images'); fig.tight_layout(); fig.savefig(RUN_DIR/'test_confusion_matrix.png',dpi=180); plt.show()
    summary={'accuracy':float(accuracy_score(y_true,y_pred)),
       'macro_f1':float(f1_score(y_true,y_pred,average='macro',zero_division=0)),
       'weighted_f1':float(f1_score(y_true,y_pred,average='weighted',zero_division=0)),
       'test_images':len(y_true),'test_manifest_sha256':EXPECTED_MANIFEST_SHA256['test.csv'],
       'test_used_for_training_or_tuning':False,'test_evaluation_passes':1,
       'parameters':sum(p.numel() for p in final_model.parameters()),
       'checkpoint_bytes':(RUN_DIR/'best_model.pt').stat().st_size,
       'latency_mean_ms':float(np.mean(latencies)),'latency_median_ms':float(np.median(latencies)),
       'latency_p95_ms':float(np.percentile(latencies,95)),'latency_batch_size':1}
    (RUN_DIR/'final_test_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print(summary)
else:
    print('Final test evaluation not started. Keep the test split frozen; enable only once for final reporting.')

## 6. Optional exports (explicit opt-in)

Exports use the best validation-selected checkpoint. ONNX and TorchScript are written to the MobileNet experiment directory. No exports are created until `RUN_EXPORTS=True`.

In [ ]:
if RUN_EXPORTS:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','-q','onnx','onnxscript'])
    best_path=RUN_DIR/'best_model.pt'
    assert best_path.is_file(), f'Train first; checkpoint not found: {best_path}'
    checkpoint=torch.load(best_path,map_location=device,weights_only=True)
    export_model=build_model(pretrained=False).to(device).eval()
    export_model.load_state_dict(checkpoint['model_state_dict'])
    dummy=torch.zeros(1,3,IMG_SIZE,IMG_SIZE,device=device)
    onnx_path=RUN_DIR/'mobilenet_v3_large.onnx'
    torch.onnx.export(export_model,dummy,str(onnx_path),input_names=['images'],output_names=['logits'],opset_version=18,dynamo=True)
    import onnx
    onnx.checker.check_model(str(onnx_path))
    torchscript_path=RUN_DIR/'mobilenet_v3_large.torchscript.pt'
    torch.jit.save(torch.jit.trace(export_model,dummy),str(torchscript_path))
    print({'onnx_bytes':onnx_path.stat().st_size,'torchscript_bytes':torchscript_path.stat().st_size})
else:
    print('Exports not started. Set RUN_EXPORTS=True after a completed training run.')

## Expected experiment artifacts

Saved beneath `MyDrive/Krishok_AI/AI/experiments/MobileNetV3_Rice/`: dataset integrity report; `best_model.pt` and `last_model.pt`; training history and train/validation loss curves; validation classification report, accuracy, macro precision/recall/F1, weighted F1, confusion matrix, model parameter count and checkpoint size; separate CPU and GPU inference latency records; environment details; optional frozen test report; and optional ONNX/TorchScript exports. No training or evaluation result is fabricated in this notebook.